In [ ]:
import pandas as pd
import json
import boto3
import vaex

# this section goes into a lambda function
s3 = boto3.Session(profile_name='personal').client('s3')


def handler(event):
    body = json.loads(event['body'])
    idf, words2id, jobs = get_files()
    words = pd.read_parquet('./seek/words-sm.parquet')
    for key in body:
        jobs = jobs[jobs[key] == body[key]]
    ids = jobs['id']
    id_filter = ids.to_frame().set_index('id')
    filtered_words = words.merge(id_filter, on='id', how='inner').drop(columns='id')
    freq = filtered_words.groupby('word_id').agg({'count': 'sum'})
    freq = freq.merge(words2id, on='word_id')
    freq = freq.set_index('word')['count']
    idf = idf.set_index('word')['count']
    df_idf = (freq * idf).dropna()
    body = (df_idf
        .sort_values(ascending=False)[:50]
        .to_frame()
        .reset_index()
        .to_numpy()
        .tolist())
    return {
        'headers': {
            'Access-Control-Allow-Origin': '*',
        },
        'statusCode': 200, 
        'body': json.dumps(body)
    }



def get_files():
    filenames = ['idf.parquet', 'words2id.parquet', 'jobs.parquet']
    return [download(name) for name in filenames]


def download(name):
    s3.download_file('jobs--001', name, f'./{name}')
    return pd.read_parquet(f'./{name}')
    

handler({'body': json.dumps({'sector': 'Information & Communication Technology'})})